In [2]:
import sqlite3
import random
from datetime import date, datetime, timedelta

DB = "airline.db"
random.seed(42)

airports = {"DEL": "Delhi", "BOM": "Mumbai", "BLR": "Bengaluru", "HYD": "Hyderabad",
            "MAA": "Chennai", "CCU": "Kolkata", "GOI": "Goa"}

# (origin, destination, base duration in minutes, base price in INR)
routes = [("DEL", "BOM", 130, 5200), ("DEL", "BLR", 165, 6100), ("BOM", "BLR", 95, 4100),
          ("DEL", "GOI", 150, 5800), ("BOM", "HYD", 85, 3900), ("DEL", "CCU", 135, 5000)]

departure_times = ["06:00", "08:30", "11:15", "14:45", "18:30", "21:50"]

In [3]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS airports (code TEXT PRIMARY KEY, city TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS flights (
    flight_id TEXT PRIMARY KEY, flight_no TEXT, origin TEXT, destination TEXT,
    departure TEXT, arrival TEXT, duration_min INTEGER, price REAL, seats_available INTEGER);
CREATE TABLE IF NOT EXISTS bookings (
    pnr TEXT PRIMARY KEY, flight_id TEXT, passenger_name TEXT, passenger_email TEXT,
    seats INTEGER, total_price REAL, status TEXT, ticket_no TEXT, created_at TEXT);
CREATE TABLE IF NOT EXISTS support_tickets (
    ticket_id TEXT PRIMARY KEY, pnr TEXT, category TEXT, description TEXT,
    status TEXT, created_at TEXT);
"""


In [4]:
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.executescript(SCHEMA)
    cursor.execute("DELETE FROM flights")
    cursor.executemany("INSERT OR REPLACE INTO airports VALUES (?, ?)", airports.items())

    rows = []
    flight_num = 100
    for origin, dest, duration, base_price in routes:
        for o, d in [(origin, dest), (dest, origin)]:
            for t in departure_times:
                flight_num += 1
                flight_no = f"FA{flight_num}"
                for offset in range(0, 8):
                    day = date.today() + timedelta(days=offset)
                    dep = datetime.combine(day, datetime.strptime(t, "%H:%M").time())
                    mins = duration + random.choice([-10, 0, 10, 20])
                    arr = dep + timedelta(minutes=mins)
                    price = round(base_price * random.uniform(0.8, 1.4), -1)
                    rows.append((f"{flight_no}-{day:%Y%m%d}", flight_no, o, d,
                                 dep.isoformat(timespec="minutes"), arr.isoformat(timespec="minutes"),
                                 mins, price, random.randint(0, 40)))
    cursor.executemany("INSERT INTO flights VALUES (?,?,?,?,?,?,?,?,?)", rows)
    conn.commit()

print(f"Seeded {len(rows)} flights")


Seeded 576 flights
